# Self-Practice Exercise – Regression Models on MovieLens

In this exercise, you will practice building and evaluating regression models using the MovieLens dataset.

## Objective
Build regression models to predict the average rating of a movie from its features.

## Models to Implement
1. Linear Regression
2. Ridge Regression (regularization)
3. Lasso Regression (feature selection)

## Dataset
MovieLens dataset (ml-latest-small)
- Download: http://files.grouplens.org/datasets/movielens/ml-latest-small.zip

## Step 1 – Download & Load the Data

Use Pandas to read the ratings and movies data directly from the ZIP file.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import requests
from zipfile import ZipFile
from io import BytesIO
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

In [ ]:
# Download and extract the MovieLens dataset
url = 'http://files.grouplens.org/datasets/movielens/ml-latest-small.zip'
print(f"Downloading dataset from {url}...")

r = requests.get(url)
z = ZipFile(BytesIO(r.content))

# Read the CSV files from the ZIP
ratings = pd.read_csv(z.open('ml-latest-small/ratings.csv'))
movies = pd.read_csv(z.open('ml-latest-small/movies.csv'))

print("\nDataset downloaded and extracted successfully!")
print(f"Ratings shape: {ratings.shape}")
print(f"Movies shape: {movies.shape}")

In [ ]:
# Preview ratings
print("First 5 rows of RATINGS:")
ratings.head()

In [ ]:
# Preview movies
print("First 5 rows of MOVIES:")
movies.head()

## Step 2 – Data Preparation

1. Merge ratings and movies on movieId
2. Convert timestamp into datetime and extract year and month
3. Aggregate at the movie level
4. One-hot encode genres into dummy variables

### 2.1 Merge ratings and movies

In [ ]:
# Merge ratings with movies
merged_data = pd.merge(ratings, movies, on='movieId')

print(f"Merged data shape: {merged_data.shape}")
merged_data.head()

### 2.2 Convert timestamp to datetime and extract features

In [ ]:
# Convert timestamp to datetime
merged_data['datetime'] = pd.to_datetime(merged_data['timestamp'], unit='s')
merged_data['year'] = merged_data['datetime'].dt.year
merged_data['month'] = merged_data['datetime'].dt.month

print("Datetime features created:")
merged_data[['timestamp', 'datetime', 'year', 'month']].head()

### 2.3 Aggregate at the movie level

In [ ]:
# Calculate average rating and number of ratings per movie
movie_aggregated = merged_data.groupby(['movieId', 'title', 'genres']).agg({
    'rating': ['mean', 'count']
}).reset_index()

# Flatten column names
movie_aggregated.columns = ['movieId', 'title', 'genres', 'avg_rating', 'num_ratings']

print(f"Aggregated movie data shape: {movie_aggregated.shape}")
movie_aggregated.head(10)

In [ ]:
# Check statistics
print("Average Rating Statistics:")
print(movie_aggregated['avg_rating'].describe())
print("\nNumber of Ratings Statistics:")
print(movie_aggregated['num_ratings'].describe())

### 2.4 One-hot encode genres

In [ ]:
# Expand genres (pipe-separated) into individual genre columns
# First, get all unique genres
all_genres = set()
for genres in movie_aggregated['genres']:
    all_genres.update(genres.split('|'))

all_genres = sorted(list(all_genres))
print(f"Unique genres: {all_genres}")

In [ ]:
# Create binary columns for each genre
for genre in all_genres:
    movie_aggregated[f'genre_{genre}'] = movie_aggregated['genres'].apply(
        lambda x: 1 if genre in x.split('|') else 0
    )

print(f"\nGenre columns created: {len(all_genres)}")
print("\nSample of genre encoding:")
movie_aggregated[['title', 'genres'] + [f'genre_{g}' for g in all_genres[:5]]].head()

In [ ]:
# Check final dataset
print(f"Final aggregated data shape: {movie_aggregated.shape}")
print(f"Columns: {movie_aggregated.columns.tolist()}")

## Step 3 – Prepare Features and Target for Modeling

Separate features (X) and target variable (y)

In [ ]:
# Define feature columns
# Features: num_ratings + all genre columns
genre_cols = [col for col in movie_aggregated.columns if col.startswith('genre_')]
feature_cols = ['num_ratings'] + genre_cols

# Prepare X (features) and y (target)
X = movie_aggregated[feature_cols]
y = movie_aggregated['avg_rating']

print(f"Feature matrix shape: {X.shape}")
print(f"Target variable shape: {y.shape}")
print(f"\nFeatures used: {len(feature_cols)}")
print(f"Feature names: {feature_cols}")

In [ ]:
# Preview X and y
print("First 5 rows of features (X):")
print(X.head())
print("\nFirst 5 values of target (y):")
print(y.head())

### Train-Test Split

In [ ]:
# Import sklearn libraries
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Sklearn libraries imported!")

In [ ]:
# Split data into train and test sets (80-20 split)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")

In [ ]:
# Scale features for better performance
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features scaled using StandardScaler")
print(f"Original feature range: [{X_train.min().min():.2f}, {X_train.max().max():.2f}]")
print(f"Scaled feature range: [{X_train_scaled.min():.2f}, {X_train_scaled.max():.2f}]")

## Step 4 – Build Regression Models

### Model 1: Linear Regression

In [ ]:
# Train Linear Regression model
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

# Make predictions
y_train_pred_lr = lr_model.predict(X_train_scaled)
y_test_pred_lr = lr_model.predict(X_test_scaled)

# Calculate metrics
r2_train_lr = r2_score(y_train, y_train_pred_lr)
r2_test_lr = r2_score(y_test, y_test_pred_lr)
rmse_train_lr = np.sqrt(mean_squared_error(y_train, y_train_pred_lr))
rmse_test_lr = np.sqrt(mean_squared_error(y_test, y_test_pred_lr))

print("Linear Regression Results:")
print("=" * 50)
print(f"Train R² Score: {r2_train_lr:.4f}")
print(f"Test R² Score: {r2_test_lr:.4f}")
print(f"Train RMSE: {rmse_train_lr:.4f}")
print(f"Test RMSE: {rmse_test_lr:.4f}")

### Model 2: Ridge Regression (L2 Regularization)

In [ ]:
# Train Ridge Regression model
ridge_model = Ridge(alpha=1.0, random_state=42)
ridge_model.fit(X_train_scaled, y_train)

# Make predictions
y_train_pred_ridge = ridge_model.predict(X_train_scaled)
y_test_pred_ridge = ridge_model.predict(X_test_scaled)

# Calculate metrics
r2_train_ridge = r2_score(y_train, y_train_pred_ridge)
r2_test_ridge = r2_score(y_test, y_test_pred_ridge)
rmse_train_ridge = np.sqrt(mean_squared_error(y_train, y_train_pred_ridge))
rmse_test_ridge = np.sqrt(mean_squared_error(y_test, y_test_pred_ridge))

print("Ridge Regression Results:")
print("=" * 50)
print(f"Train R² Score: {r2_train_ridge:.4f}")
print(f"Test R² Score: {r2_test_ridge:.4f}")
print(f"Train RMSE: {rmse_train_ridge:.4f}")
print(f"Test RMSE: {rmse_test_ridge:.4f}")

### Model 3: Lasso Regression (L1 Regularization / Feature Selection)

In [ ]:
# Train Lasso Regression model
lasso_model = Lasso(alpha=0.01, random_state=42)
lasso_model.fit(X_train_scaled, y_train)

# Make predictions
y_train_pred_lasso = lasso_model.predict(X_train_scaled)
y_test_pred_lasso = lasso_model.predict(X_test_scaled)

# Calculate metrics
r2_train_lasso = r2_score(y_train, y_train_pred_lasso)
r2_test_lasso = r2_score(y_test, y_test_pred_lasso)
rmse_train_lasso = np.sqrt(mean_squared_error(y_train, y_train_pred_lasso))
rmse_test_lasso = np.sqrt(mean_squared_error(y_test, y_test_pred_lasso))

print("Lasso Regression Results:")
print("=" * 50)
print(f"Train R² Score: {r2_train_lasso:.4f}")
print(f"Test R² Score: {r2_test_lasso:.4f}")
print(f"Train RMSE: {rmse_train_lasso:.4f}")
print(f"Test RMSE: {rmse_test_lasso:.4f}")

## Step 5 – Model Evaluation & Comparison

Create a comparison table summarizing all models

In [ ]:
# Create comparison DataFrame
comparison_df = pd.DataFrame({
    'Model': ['Linear Regression', 'Ridge Regression', 'Lasso Regression'],
    'Train R²': [r2_train_lr, r2_train_ridge, r2_train_lasso],
    'Test R²': [r2_test_lr, r2_test_ridge, r2_test_lasso],
    'Train RMSE': [rmse_train_lr, rmse_train_ridge, rmse_train_lasso],
    'Test RMSE': [rmse_test_lr, rmse_test_ridge, rmse_test_lasso]
})

print("Model Comparison Summary:")
print("=" * 80)
comparison_df

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# R² Score Comparison
x_pos = np.arange(len(comparison_df))
width = 0.35

axes[0].bar(x_pos - width/2, comparison_df['Train R²'], width, label='Train R²', color='steelblue')
axes[0].bar(x_pos + width/2, comparison_df['Test R²'], width, label='Test R²', color='coral')
axes[0].set_xlabel('Model', fontsize=12)
axes[0].set_ylabel('R² Score', fontsize=12)
axes[0].set_title('R² Score Comparison', fontsize=14, fontweight='bold')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(comparison_df['Model'], rotation=15, ha='right')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# RMSE Comparison
axes[1].bar(x_pos - width/2, comparison_df['Train RMSE'], width, label='Train RMSE', color='steelblue')
axes[1].bar(x_pos + width/2, comparison_df['Test RMSE'], width, label='Test RMSE', color='coral')
axes[1].set_xlabel('Model', fontsize=12)
axes[1].set_ylabel('RMSE', fontsize=12)
axes[1].set_title('RMSE Comparison', fontsize=14, fontweight='bold')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(comparison_df['Model'], rotation=15, ha='right')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## Step 6 – Interpret Results

Identify which features are most predictive

### Feature Coefficients Analysis

In [ ]:
# Create DataFrame with feature coefficients
coef_df = pd.DataFrame({
    'Feature': feature_cols,
    'Linear_Coef': lr_model.coef_,
    'Ridge_Coef': ridge_model.coef_,
    'Lasso_Coef': lasso_model.coef_
})

# Add absolute values for sorting
coef_df['Linear_Abs'] = np.abs(coef_df['Linear_Coef'])
coef_df['Ridge_Abs'] = np.abs(coef_df['Ridge_Coef'])
coef_df['Lasso_Abs'] = np.abs(coef_df['Lasso_Coef'])

print("Feature Coefficients:")
print("=" * 80)
coef_df.head(10)

In [ ]:
# Top 10 most important features (by Linear Regression)
top_features_lr = coef_df.nlargest(10, 'Linear_Abs')

print("Top 10 Most Important Features (Linear Regression):")
print("=" * 80)
top_features_lr[['Feature', 'Linear_Coef', 'Ridge_Coef', 'Lasso_Coef']]

In [ ]:
# Visualize top 15 feature coefficients
top_n = 15
top_features = coef_df.nlargest(top_n, 'Linear_Abs')

fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Linear Regression coefficients
axes[0].barh(range(len(top_features)), top_features['Linear_Coef'], color='steelblue')
axes[0].set_yticks(range(len(top_features)))
axes[0].set_yticklabels(top_features['Feature'])
axes[0].set_xlabel('Coefficient Value', fontsize=12)
axes[0].set_title('Linear Regression - Top 15 Features', fontsize=14, fontweight='bold')
axes[0].grid(axis='x', alpha=0.3)
axes[0].invert_yaxis()

# Ridge Regression coefficients
axes[1].barh(range(len(top_features)), top_features['Ridge_Coef'], color='coral')
axes[1].set_yticks(range(len(top_features)))
axes[1].set_yticklabels(top_features['Feature'])
axes[1].set_xlabel('Coefficient Value', fontsize=12)
axes[1].set_title('Ridge Regression - Top 15 Features', fontsize=14, fontweight='bold')
axes[1].grid(axis='x', alpha=0.3)
axes[1].invert_yaxis()

# Lasso Regression coefficients
axes[2].barh(range(len(top_features)), top_features['Lasso_Coef'], color='teal')
axes[2].set_yticks(range(len(top_features)))
axes[2].set_yticklabels(top_features['Feature'])
axes[2].set_xlabel('Coefficient Value', fontsize=12)
axes[2].set_title('Lasso Regression - Top 15 Features', fontsize=14, fontweight='bold')
axes[2].grid(axis='x', alpha=0.3)
axes[2].invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# Count non-zero coefficients in Lasso (feature selection)
non_zero_lasso = np.sum(lasso_model.coef_ != 0)
zero_lasso = np.sum(lasso_model.coef_ == 0)

print(f"Lasso Feature Selection:")
print(f"Non-zero coefficients: {non_zero_lasso}")
print(f"Zero coefficients (features removed): {zero_lasso}")
print(f"\nFeatures retained by Lasso:")
retained_features = coef_df[coef_df['Lasso_Coef'] != 0]['Feature'].tolist()
print(retained_features)

### Prediction vs Actual Plots

In [ ]:
# Plot predictions vs actual values
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Linear Regression
axes[0].scatter(y_test, y_test_pred_lr, alpha=0.5, color='steelblue')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0].set_xlabel('Actual Rating', fontsize=12)
axes[0].set_ylabel('Predicted Rating', fontsize=12)
axes[0].set_title(f'Linear Regression\nR² = {r2_test_lr:.4f}', fontsize=14, fontweight='bold')
axes[0].grid(alpha=0.3)

# Ridge Regression
axes[1].scatter(y_test, y_test_pred_ridge, alpha=0.5, color='coral')
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[1].set_xlabel('Actual Rating', fontsize=12)
axes[1].set_ylabel('Predicted Rating', fontsize=12)
axes[1].set_title(f'Ridge Regression\nR² = {r2_test_ridge:.4f}', fontsize=14, fontweight='bold')
axes[1].grid(alpha=0.3)

# Lasso Regression
axes[2].scatter(y_test, y_test_pred_lasso, alpha=0.5, color='teal')
axes[2].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[2].set_xlabel('Actual Rating', fontsize=12)
axes[2].set_ylabel('Predicted Rating', fontsize=12)
axes[2].set_title(f'Lasso Regression\nR² = {r2_test_lasso:.4f}', fontsize=14, fontweight='bold')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Residual Analysis

In [ ]:
# Calculate residuals
residuals_lr = y_test - y_test_pred_lr
residuals_ridge = y_test - y_test_pred_ridge
residuals_lasso = y_test - y_test_pred_lasso

# Plot residuals
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Linear Regression residuals
axes[0].scatter(y_test_pred_lr, residuals_lr, alpha=0.5, color='steelblue')
axes[0].axhline(y=0, color='r', linestyle='--', lw=2)
axes[0].set_xlabel('Predicted Rating', fontsize=12)
axes[0].set_ylabel('Residuals', fontsize=12)
axes[0].set_title('Linear Regression - Residual Plot', fontsize=14, fontweight='bold')
axes[0].grid(alpha=0.3)

# Ridge Regression residuals
axes[1].scatter(y_test_pred_ridge, residuals_ridge, alpha=0.5, color='coral')
axes[1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1].set_xlabel('Predicted Rating', fontsize=12)
axes[1].set_ylabel('Residuals', fontsize=12)
axes[1].set_title('Ridge Regression - Residual Plot', fontsize=14, fontweight='bold')
axes[1].grid(alpha=0.3)

# Lasso Regression residuals
axes[2].scatter(y_test_pred_lasso, residuals_lasso, alpha=0.5, color='teal')
axes[2].axhline(y=0, color='r', linestyle='--', lw=2)
axes[2].set_xlabel('Predicted Rating', fontsize=12)
axes[2].set_ylabel('Residuals', fontsize=12)
axes[2].set_title('Lasso Regression - Residual Plot', fontsize=14, fontweight='bold')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Step 7 – Extensions (Optional, Advanced 🌟)

### Extension 1: Decision Tree Regressor

In [ ]:
# Import Decision Tree
from sklearn.tree import DecisionTreeRegressor, plot_tree

# Train Decision Tree model
dt_model = DecisionTreeRegressor(max_depth=5, random_state=42)
dt_model.fit(X_train, y_train)  # Note: using unscaled data for tree models

# Make predictions
y_train_pred_dt = dt_model.predict(X_train)
y_test_pred_dt = dt_model.predict(X_test)

# Calculate metrics
r2_train_dt = r2_score(y_train, y_train_pred_dt)
r2_test_dt = r2_score(y_test, y_test_pred_dt)
rmse_train_dt = np.sqrt(mean_squared_error(y_train, y_train_pred_dt))
rmse_test_dt = np.sqrt(mean_squared_error(y_test, y_test_pred_dt))

print("Decision Tree Regressor Results:")
print("=" * 50)
print(f"Train R² Score: {r2_train_dt:.4f}")
print(f"Test R² Score: {r2_test_dt:.4f}")
print(f"Train RMSE: {rmse_train_dt:.4f}")
print(f"Test RMSE: {rmse_test_dt:.4f}")

In [ ]:
# Visualize Decision Tree
plt.figure(figsize=(20, 10))
plot_tree(dt_model, 
          feature_names=feature_cols, 
          filled=True, 
          rounded=True,
          fontsize=10)
plt.title('Decision Tree Visualization (max_depth=5)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance from Decision Tree
feature_importance = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': dt_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("Top 10 Feature Importances (Decision Tree):")
print("=" * 50)
print(feature_importance.head(10))

In [ ]:
# Visualize feature importance
top_features_dt = feature_importance.head(15)

plt.figure(figsize=(12, 8))
plt.barh(range(len(top_features_dt)), top_features_dt['Importance'], color='purple')
plt.yticks(range(len(top_features_dt)), top_features_dt['Feature'])
plt.xlabel('Importance', fontsize=12)
plt.title('Decision Tree - Top 15 Feature Importances', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

### Compare Coefficients vs Feature Importances

In [ ]:
# Compare linear model coefficients with tree feature importances
comparison_features = pd.merge(
    coef_df[['Feature', 'Linear_Abs', 'Ridge_Abs', 'Lasso_Abs']],
    feature_importance,
    on='Feature'
).sort_values('Importance', ascending=False)

print("Top 10 Features - Coefficients vs Importances:")
print("=" * 80)
comparison_features.head(10)

### Final Model Comparison (Including Decision Tree)

In [ ]:
# Update comparison table
final_comparison = pd.DataFrame({
    'Model': ['Linear Regression', 'Ridge Regression', 'Lasso Regression', 'Decision Tree'],
    'Train R²': [r2_train_lr, r2_train_ridge, r2_train_lasso, r2_train_dt],
    'Test R²': [r2_test_lr, r2_test_ridge, r2_test_lasso, r2_test_dt],
    'Train RMSE': [rmse_train_lr, rmse_train_ridge, rmse_train_lasso, rmse_train_dt],
    'Test RMSE': [rmse_test_lr, rmse_test_ridge, rmse_test_lasso, rmse_test_dt]
})

print("Final Model Comparison (All Models):")
print("=" * 80)
final_comparison

## Summary and Key Insights

### Model Performance
- All models show similar performance on this dataset
- Linear models benefit from feature scaling
- Lasso performs feature selection by setting some coefficients to zero
- Decision Tree can capture non-linear relationships

### Key Predictive Features
1. **Number of ratings**: Movies with more ratings tend to have more reliable average ratings
2. **Genre effects**: Certain genres (e.g., Film-Noir, War, Documentary) tend to have higher/lower ratings
3. **Regularization impact**: Ridge and Lasso help prevent overfitting

### Model Selection
- **Linear Regression**: Good baseline, interpretable coefficients
- **Ridge Regression**: Handles multicollinearity better
- **Lasso Regression**: Automatic feature selection
- **Decision Tree**: Can model non-linear patterns, provides feature importance

### Potential Improvements
1. Add user-level features (average rating given by each user)
2. Try ensemble methods (Random Forest, Gradient Boosting)
3. Use cross-validation for hyperparameter tuning
4. Predict individual ratings instead of movie-level averages
5. Implement time-based splits for temporal validation

## Exercise Complete!

You have successfully:
1. ✓ Downloaded and loaded the MovieLens dataset
2. ✓ Prepared features (aggregation, one-hot encoding)
3. ✓ Built three regression models (Linear, Ridge, Lasso)
4. ✓ Evaluated models with R² and RMSE metrics
5. ✓ Interpreted feature importance
6. ✓ Extended with Decision Tree regressor

Great job completing this regression exercise! 🎉